# (선택) 코드로 파인튜닝하기 — GPU 필요
강의장에서는 **Unsloth Studio**(화면 클릭)로 학습해요. 이 노트북은 같은 설정을 코드로 돌리는 **대체 방법**이에요.

| 어디서 | 준비 |
|---|---|
| Colab | 런타임 → 런타임 유형 변경 → **T4 GPU** |
| 내 컴퓨터 | NVIDIA GPU + Unsloth 설치 (RTX 50 시리즈는 CUDA 12.8 이상 PyTorch 필요) |

준비물: `train.jsonl`, `eval.jsonl`, `test.jsonl` (split_data.ipynb로 만든 파일) — 같은 폴더에 두기
⚠️ 학습한 모델·GGUF 파일은 온라인(GitHub 등)에 올리지 않아요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

In [ ]:
assert GPU, 'GPU가 없어요. Colab은 T4 GPU 런타임으로 바꾸고, 내 컴퓨터는 Unsloth Studio를 쓰세요.'

In [ ]:
# 설치 (Colab은 매번, 내 컴퓨터는 한 번만) — 몇 분 걸려요
%pip install -q unsloth

## 1. 기본 모델 불러오기 — 4bit로 작게 (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

## 2. 데이터를 대화 형식으로 바꾸기

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files={'train': 'train.jsonl', 'eval': 'eval.jsonl'})

def to_text(ex):
    msgs = [{'role': 'user', 'content': ex['instruction'] + '\n\n' + ex['input']},
            {'role': 'assistant', 'content': ex['output']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = ds.map(to_text)
print(ds)

## 3. 학습 — Studio와 같은 설정 (에포크 2 · 배치 2×누적 4 · LR 2e-4)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds['train'], eval_dataset=ds['eval'],
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=2048,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4,
        optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear',
        logging_steps=1, eval_strategy='epoch',
        seed=42, output_dir='outputs', report_to='none',
    ),
)
stats = trainer.train()

In [ ]:
# loss 그래프 — train은 내려가는데 eval이 올라가면 외우기 시작(과적합)
import pandas as pd
log = pd.DataFrame(trainer.state.log_history)
ax = log.dropna(subset=['loss']).plot(x='step', y='loss', label='train', figsize=(8, 4))
if 'eval_loss' in log:
    log.dropna(subset=['eval_loss']).plot(x='step', y='eval_loss', label='eval', ax=ax, marker='o')
ax.set_ylabel('loss'); plt.show()

## 4. 시험 문제로 확인 — 본문에 없는 숫자·원인을 지어내지 않았나?

In [ ]:
FastLanguageModel.for_inference(model)
test = pd.read_json('test.jsonl', lines=True)
for _, r in test.iterrows():
    msgs = [{'role': 'user', 'content': r['instruction'] + '\n\n' + r['input']}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False)
    print('모델:', tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print('정답:', r['output']); print('-' * 40)

## 5. (더 해보기) GGUF로 내보내 Ollama에서 쓰기

In [ ]:
model.save_pretrained_gguf('lead_model', tokenizer, quantization_method='q4_k_m')
# 만들어진 .gguf 파일을 내려받아 Modelfile 첫 줄에  FROM ./파일이름.gguf
# 터미널: ollama create 내모델이름 -f Modelfile